# Version 1 versus Version 2: Shared 2024 Benchmark

This notebook compares the locked TF-IDF + Linear SVM benchmark with the frozen DistilBERT challenger on the same 6,609-row final internal test.

The Version 2 routing policy is selected, saved, fingerprinted, and reloaded from development OOF outputs before the cleaned CSV is parsed. Neither model is trained or modified. Row-level outputs stay under the Git-ignored comparison directory; 2025 and 2026 data are never accessed. Transformer softmax scores and margins are uncalibrated model signals, not probabilities.


## 1. Environment, immutable controls, and timing protocol

Timing is precommitted to the same first 128 benchmark texts, batch size 16, excluded warm-ups, repeated measurements, median latency, and tokenization included for DistilBERT.


In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, json, os, platform, subprocess, sys, time, warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import sklearn
import torch
import transformers
from scipy.special import softmax
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"

def find_root(start):
    p=start.resolve()
    for c in (p,*p.parents):
        if (c/".gitignore").is_file() and (c/"docs/v2_experiment_plan.md").is_file():
            return c
    raise FileNotFoundError("Repository root not found")
ROOT=find_root(Path.cwd())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.routing_rules import AUTO_ROUTE, route_from_scores

DATA=ROOT/"data/processed/cfpb_complaints_2024_cleaned.csv"
V1_PATH=ROOT/"models/best_tfidf_classifier.joblib"
V2_DIR=ROOT/"models/v2_distilbert_challenger/final"
OOF_PATH=ROOT/"models/v2_distilbert_challenger/oof/development_oof_outputs.npz"
LOCAL=ROOT/"models/v1_v2_2024_comparison"
POLICY_PATH=LOCAL/"v2_routing_policy.json"
ROW_PATH=LOCAL/"final_test_outputs.npz"
SUMMARY_PATH=LOCAL/"comparison_summary.json"
FIG_DIR=ROOT/"reports/figures"
CM_FIG=FIG_DIR/"v1_v2_2024_confusion_matrices.png"
ROUTING_FIG=FIG_DIR/"v1_v2_2024_routing_comparison.png"
COMPUTE_FIG=FIG_DIR/"v1_v2_2024_compute_comparison.png"
NOTEBOOK_PATH=ROOT/"notebooks/10_v1_v2_champion_challenger_comparison.ipynb"

SOURCE_SIZE=54_908_639
SOURCE_SHA="b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919"
V1_SIZE=3_392_109
V1_SHA="4514e7e49e305e408e2eaaf296d8607b33e9320547685339eff263e4dda0c94a"
OOF_SIZE=674_788
OOF_SHA="72d59db97819d6f06b968520eddac2d0c1d590f36dea4efa627e9c123c1e5b13"
V2_FILES={
"config.json":(1229,"745d87e88a54bd5bd349b14aae194a1f523189cb6d6f377463419487fa43e370"),
"model.safetensors":(267851024,"e05900579f16e96d75df968cedb71b2b2fde3aae95f1bf73dbe7147306287c23"),
"special_tokens_map.json":(132,"3c3507f36dff57bce437223db3b3081d1e2b52ec3e56ee55438193ecb2c94dd6"),
"tokenizer.json":(711494,"8b79639ec74b46604e730f505186eaafb1006d2fd00f2c4930d168bb7f894680"),
"tokenizer_config.json":(1283,"21c3bea73b6711617c657664adcd4d0b02ce20d2db4b2ebdd722a6da8da28bcd"),
"training_args.bin":(6033,"568405730a290d2928d67b32870a9a965253e46bc2e9ed516048100a9ba475d8"),
"training_summary.json":(3329,"b8a149005ac05614230937c3620fcda2162b9f39c126a1aefa95ef8721a0bfcf"),
"vocab.txt":(231508,"07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3")}
LABELS=("Checking or savings account","Credit card","Credit reporting or other personal consumer reports","Debt collection","Money transfer, virtual currency, or money service","Mortgage","Student loan","Vehicle loan or lease")
SHORT=("Checking / savings","Credit card","Credit reporting","Debt collection","Money transfer","Mortgage","Student loan","Vehicle loan / lease")
label2id={x:i for i,x in enumerate(LABELS)}
id2label={i:x for x,i in label2id.items()}
V1_TOP,V1_MARGIN=0.08,0.73
V1_REF={"accuracy":.8712,"macro_precision":.7734,"macro_recall":.7621,"macro_f1":.7671,"weighted_precision":.8721,"weighted_recall":.8712,"weighted_f1":.8715,"coverage":.7705,"routed_accuracy":.9503,"misroute_rate":.0497}
Q=np.linspace(0,.975,40)
RISK_LIMITS=(.03,.05,.075,.10)
COVERAGE_TARGETS=(.25,.50,.75)
TIMING_N,BATCH=128,16
SINGLE_WARM,SINGLE_REPS,BATCH_WARM,BATCH_REPS=2,10,1,3
FINAL_TEST_DATA_LOADED=False
ACCESSED=set()
expected_env={"Python":"3.11.15","scikit-learn":"1.9.0","PyTorch":"2.9.1+cu126","Transformers":"4.57.6"}
observed_env={"Python":platform.python_version(),"scikit-learn":sklearn.__version__,"PyTorch":torch.__version__,"Transformers":transformers.__version__}
if Path(sys.prefix).name!="complaint-v2" or observed_env!=expected_env: raise RuntimeError("Environment mismatch")
if not torch.cuda.is_available() or torch.cuda.get_device_name(0)!="NVIDIA GeForce GTX 1650": raise RuntimeError("CUDA/GPU mismatch")
branch=subprocess.check_output(["git","branch","--show-current"],cwd=ROOT,text=True).strip()
if branch not in {"v2/issue-4-champion-challenger-comparison","main"}: raise RuntimeError(branch)
print("Environment and immutable controls: PASS")
print(observed_env)
print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
print("Timing: 128 shared texts; batch 16; warm-ups excluded; repeated medians.")


Environment and immutable controls: PASS
{'Python': '3.11.15', 'scikit-learn': '1.9.0', 'PyTorch': '2.9.1+cu126', 'Transformers': '4.57.6'}
CUDA GPU: NVIDIA GeForce GTX 1650
Timing: 128 shared texts; batch 16; warm-ups excluded; repeated medians.


## 2. OOF-only policy selection and lock

This section validates all fingerprints, selects the policy only from the saved 26,433-row development OOF artifact, writes a local policy record, fingerprints it, and reloads it. The CSV has not yet been parsed.


In [2]:
def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def ignored(path):
    return subprocess.run(["git","check-ignore","-q",path.relative_to(ROOT).as_posix()],cwd=ROOT,check=False).returncode==0

def class_metrics(y,p):
    ma=precision_recall_fscore_support(y,p,labels=np.arange(8),average="macro",zero_division=0)
    we=precision_recall_fscore_support(y,p,labels=np.arange(8),average="weighted",zero_division=0)
    return {"accuracy":float(accuracy_score(y,p)),"macro_precision":float(ma[0]),"macro_recall":float(ma[1]),"macro_f1":float(ma[2]),"weighted_precision":float(we[0]),"weighted_recall":float(we[1]),"weighted_f1":float(we[2])}

def category_class(y,p):
    pr,re,f1,s=precision_recall_fscore_support(y,p,labels=np.arange(8),average=None,zero_division=0)
    return pd.DataFrame({"Category":LABELS,"Precision":pr,"Recall":re,"F1":f1,"Support":s.astype(int)})

def route_summary(y,p,a):
    n=len(y); r=int(a.sum()); c=int(((y==p)&a).sum()); bad=r-c
    return {"rows":int(n),"auto_routed":r,"human_review":int(n-r),"coverage":float(r/n),"review_rate":float(1-r/n),"correct_routed":c,"incorrect_routed":bad,"routed_accuracy":float(c/r) if r else None,"misroute_rate":float(bad/r) if r else None}

def category_route(y,p,a):
    rows=[]
    for i,name in enumerate(LABELS):
        mask=y==i; routed=mask&a; s=int(mask.sum()); r=int(routed.sum()); c=int(((y==p)&routed).sum())
        acc=float(c/r) if r else np.nan
        rows.append({"Category":name,"Support":s,"Auto-routed":r,"Coverage":float(r/s),"Review rate":float(1-r/s),"Routed accuracy":acc,"Misroute rate":float(1-acc) if r else np.nan})
    return pd.DataFrame(rows)

checks={"source_size":DATA.stat().st_size==SOURCE_SIZE,"source_hash":sha(DATA)==SOURCE_SHA,"v1_size":V1_PATH.stat().st_size==V1_SIZE,"v1_hash":sha(V1_PATH)==V1_SHA,"oof_size":OOF_PATH.stat().st_size==OOF_SIZE,"oof_hash":sha(OOF_PATH)==OOF_SHA}
for n,(s,h) in V2_FILES.items(): checks[f"v2_{n}"]=(V2_DIR/n).stat().st_size==s and sha(V2_DIR/n)==h
with warnings.catch_warnings(record=True) as ws:
    warnings.simplefilter("always"); probe_v1=joblib.load(V1_PATH)
checks["v1_reload"]=len(ws)==0 and tuple(probe_v1.classes_)==LABELS
del probe_v1
probe_v2=DistilBertForSequenceClassification.from_pretrained(V2_DIR,local_files_only=True)
probe_tok=DistilBertTokenizerFast.from_pretrained(V2_DIR,local_files_only=True)
checks["v2_reload"]=probe_v2.config.num_labels==8 and probe_v2.config.label2id==label2id and probe_v2.config.id2label==id2label and type(probe_tok).__name__=="DistilBertTokenizerFast"
checks["v2_finite"]=all(torch.isfinite(p).all().item() for p in probe_v2.parameters())
del probe_v2,probe_tok; gc.collect()
if not all(checks.values()): raise RuntimeError([k for k,v in checks.items() if not v])

with np.load(OOF_PATH,allow_pickle=False) as z:
    oof_y=z["true_labels"].astype(np.int64); oof_p=z["predicted_labels"].astype(np.int64)
    oof_logits=z["logits"].astype(np.float32); oof_fold=z["fold_assignment"].astype(np.int8)
    oof_top=z["top_softmax_score"].astype(np.float64); oof_margin=z["top_two_softmax_margin"].astype(np.float64)
if oof_y.shape!=(26433,) or oof_logits.shape!=(26433,8) or set(np.unique(oof_fold))!=set(range(5)): raise RuntimeError("OOF shape/fold mismatch")
if not all(np.isfinite(x).all() for x in (oof_y,oof_p,oof_logits,oof_top,oof_margin)): raise RuntimeError("OOF non-finite")
if not np.array_equal(oof_p,np.argmax(oof_logits,axis=1)): raise RuntimeError("OOF prediction mismatch")
tops=sorted(set(np.round(np.quantile(oof_top,Q),2).tolist()))
margins=sorted(set(np.round(np.quantile(oof_margin,Q),2).tolist()))
qualifying=[]
for t in tops:
    for m in margins:
        auto=(oof_top>=t)&(oof_margin>=m)&(oof_margin>0); routed=int(auto.sum())
        if not routed: continue
        cov=float(routed/len(oof_y)); risk=float(((oof_y!=oof_p)&auto).sum()/routed)
        if cov>=.05 and risk<=.05:
            qualifying.append({"top_score_threshold":float(t),"margin_threshold":float(m),"coverage":cov,"routed_accuracy":float(1-risk),"misroute_rate":risk,"auto_routed":routed,"human_review":int(len(oof_y)-routed)})
if not qualifying: raise RuntimeError("No qualifying policy")
qualifying.sort(key=lambda r:(-r["coverage"],r["misroute_rate"],r["top_score_threshold"],r["margin_threshold"]))
selected=qualifying[0]
if FINAL_TEST_DATA_LOADED: raise RuntimeError("Final test loaded before policy lock")
LOCAL.mkdir(parents=True,exist_ok=True)
policy={"created_utc":datetime.now(timezone.utc).isoformat(),"oof_artifact":OOF_PATH.relative_to(ROOT).as_posix(),"oof_size_bytes":OOF_SIZE,"oof_sha256":OOF_SHA,"oof_rows":26433,"signals":["top_softmax_score","top_two_softmax_margin"],"interpretation":"uncalibrated model signals, not probabilities","quantile_levels":[float(x) for x in Q],"candidate_rounding_decimals":2,"top_score_candidates":[float(x) for x in tops],"margin_candidates":[float(x) for x in margins],"inclusive_thresholds":True,"positive_margin_required":True,"minimum_coverage":.05,"maximum_misroute_rate":.05,"tie_break":["highest coverage","lower misroute rate","lower top-score threshold","lower margin threshold"],"candidate_pairs_evaluated":len(tops)*len(margins),"qualifying_pairs":len(qualifying),"selected":selected,"final_test_data_loaded_at_lock":False}
POLICY_PATH.write_text(json.dumps(policy,indent=2,sort_keys=True),encoding="utf-8")
policy_sha=sha(POLICY_PATH); locked_policy=json.loads(POLICY_PATH.read_text(encoding="utf-8"))
if locked_policy!=policy or locked_policy["final_test_data_loaded_at_lock"] is not False: raise RuntimeError("Policy reload mismatch")
print(f"Artifact preflight: {sum(checks.values())}/{len(checks)} PASS")
print(f"Candidate pairs / qualifying: {len(tops)*len(margins)} / {len(qualifying)}")
print(f"Selected V2 thresholds: top >= {selected['top_score_threshold']:.2f}; margin >= {selected['margin_threshold']:.2f}; positive margin")
print(f"Development OOF: coverage {selected['coverage']:.6f}; routed accuracy {selected['routed_accuracy']:.6f}; misroute {selected['misroute_rate']:.6f}")
print(f"Policy SHA-256: {policy_sha}")
print("Final-test data loaded before policy lock: NO")


Artifact preflight: 17/17 PASS
Candidate pairs / qualifying: 304 / 170
Selected V2 thresholds: top >= 0.22; margin >= 0.91; positive margin
Development OOF: coverage 0.755003; routed accuracy 0.951446; misroute 0.048554
Policy SHA-256: 9ca16a8533f26f8e00fd9d57c654af66fa78e21880f7fa7783a9d1adf964d818
Final-test data loaded before policy lock: NO


## 3. Shared benchmark reconstruction and frozen-model evaluation

Only after the policy is locked and reloaded does this section parse the locked 2024 source. Both frozen models evaluate the same rows; Version 1 must reproduce its committed reference.


In [3]:
def norm(v):
    x=" ".join(str(v).strip().split())
    if not x: raise ValueError("Empty normalized text")
    return x
def text_hash(v): return hashlib.sha256(norm(v).encode("utf-8")).hexdigest()

if not POLICY_PATH.is_file() or sha(POLICY_PATH)!=policy_sha: raise RuntimeError("Policy unavailable")
FINAL_TEST_DATA_LOADED=True; ACCESSED.add(DATA.resolve())
df=pd.read_csv(DATA,usecols=["clean_complaint_text","product"])
if len(df)!=50000 or df.isna().any().any(): raise RuntimeError("Source mismatch")
df["source_row_order"]=np.arange(len(df),dtype=np.int64)
df["group"]=df["clean_complaint_text"].map(text_hash)
counts=df.groupby("group",sort=False)["product"].nunique()
bad=set(counts[counts>1].index)
scope=df[df["product"].isin(LABELS)]
clean=scope[~scope["group"].isin(bad)].drop_duplicates(["group","product"],keep="first").reset_index(drop=True)
splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
dev_idx,test_idx=list(splitter.split(clean["clean_complaint_text"],clean["product"],groups=clean["group"]))[0]
overlap=len(set(clean.iloc[dev_idx]["group"])&set(clean.iloc[test_idx]["group"]))
test=clean.iloc[test_idx].reset_index(drop=True)
if len(clean)!=33042 or len(dev_idx)!=26433 or len(test)!=6609 or overlap!=0 or set(test["product"])!=set(LABELS): raise RuntimeError("Benchmark reconstruction mismatch")
texts=test["clean_complaint_text"].tolist(); y=test["product"].map(label2id).to_numpy(dtype=np.int64)
del df,scope,clean,test,bad,counts; gc.collect()
print("Benchmark: 33,042 corrected; 26,433 development; 6,609 final test; zero overlap; eight classes: PASS")

proc=psutil.Process()
rss0=proc.memory_info().rss; start=time.perf_counter()
with warnings.catch_warnings(record=True) as ws:
    warnings.simplefilter("always"); v1=joblib.load(V1_PATH)
v1_load=time.perf_counter()-start; rss1=proc.memory_info().rss
if ws or tuple(v1.classes_)!=LABELS: raise RuntimeError("V1 load mismatch")
v1_names=v1.predict(texts); v1_p=np.asarray([label2id[x] for x in v1_names],dtype=np.int64)
v1_scores=np.asarray(v1.decision_function(texts),dtype=np.float64)
routes=[route_from_scores(v1.classes_,s,min_top_score=.08,min_score_margin=.73) for s in v1_scores]
v1_auto=np.asarray([r["routing_decision"]==AUTO_ROUTE for r in routes],dtype=bool)
v1_top=np.asarray([r["top_score"] for r in routes]); v1_margin=np.asarray([r["score_margin"] for r in routes])
v1_m=class_metrics(y,v1_p); v1_r=route_summary(y,v1_p,v1_auto)
v1_cc=category_class(y,v1_p); v1_cr=category_route(y,v1_p,v1_auto)
v1_cm=confusion_matrix(y,v1_p,labels=np.arange(8)); v1_cmn=v1_cm/v1_cm.sum(axis=1,keepdims=True)
for n in ("accuracy","macro_precision","macro_recall","macro_f1","weighted_precision","weighted_recall","weighted_f1"):
    if round(v1_m[n],4)!=V1_REF[n]: raise RuntimeError(f"V1 metric mismatch: {n}")
for n in ("coverage","routed_accuracy","misroute_rate"):
    if round(v1_r[n],4)!=V1_REF[n]: raise RuntimeError(f"V1 routing mismatch: {n}")

rss2=proc.memory_info().rss; start=time.perf_counter()
tok=DistilBertTokenizerFast.from_pretrained(V2_DIR,local_files_only=True)
v2=DistilBertForSequenceClassification.from_pretrained(V2_DIR,local_files_only=True)
v2_load=time.perf_counter()-start; rss3=proc.memory_info().rss
if v2.config.label2id!=label2id or v2.config.id2label!=id2label or not all(torch.isfinite(p).all().item() for p in v2.parameters()): raise RuntimeError("V2 reload mismatch")
device=torch.device("cuda"); v2.to(device).eval(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
parts=[]
with torch.inference_mode():
    for i in range(0,len(texts),16):
        enc=tok(texts[i:i+16],truncation=True,max_length=256,padding=True,return_tensors="pt",return_token_type_ids=False)
        enc={k:v.to(device) for k,v in enc.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16): parts.append(v2(**enc).logits.float().cpu().numpy())
eval_alloc=torch.cuda.max_memory_allocated()/(1024**2); eval_reserved=torch.cuda.max_memory_reserved()/(1024**2)
v2_logits=np.concatenate(parts).astype(np.float32)
if v2_logits.shape!=(6609,8) or not np.isfinite(v2_logits).all(): raise RuntimeError("V2 logits mismatch")
v2_p=np.argmax(v2_logits,axis=1); probs=softmax(v2_logits.astype(np.float64),axis=1); ordered=np.sort(probs,axis=1)
v2_top=ordered[:,-1]; v2_margin=ordered[:,-1]-ordered[:,-2]
t=float(locked_policy["selected"]["top_score_threshold"]); m=float(locked_policy["selected"]["margin_threshold"])
v2_auto=(v2_top>=t)&(v2_margin>=m)&(v2_margin>0)
v2_m=class_metrics(y,v2_p); v2_r=route_summary(y,v2_p,v2_auto)
v2_cc=category_class(y,v2_p); v2_cr=category_route(y,v2_p,v2_auto)
v2_cm=confusion_matrix(y,v2_p,labels=np.arange(8)); v2_cmn=v2_cm/v2_cm.sum(axis=1,keepdims=True)

np.savez_compressed(ROW_PATH,true_labels=y,v1_predicted_labels=v1_p,v1_decision_scores=v1_scores.astype(np.float32),v1_top_score=v1_top.astype(np.float32),v1_score_margin=v1_margin.astype(np.float32),v1_auto_route=v1_auto,v2_predicted_labels=v2_p,v2_logits=v2_logits,v2_top_softmax_score=v2_top.astype(np.float32),v2_top_two_softmax_margin=v2_margin.astype(np.float32),v2_auto_route=v2_auto)
row_sha=sha(ROW_PATH)
print("V1 classification:",v1_m); print("V1 routing:",v1_r)
print("V2 classification:",v2_m); print("V2 routing:",v2_r)
print(f"V2 eval GPU peak allocated/reserved MiB: {eval_alloc:.1f}/{eval_reserved:.1f}")
print(f"Local row-output SHA-256: {row_sha}")
print("No row-level outputs displayed.")


Benchmark: 33,042 corrected; 26,433 development; 6,609 final test; zero overlap; eight classes: PASS


V1 classification: {'accuracy': 0.871236193070056, 'macro_precision': 0.7734155138213924, 'macro_recall': 0.7620834294675218, 'macro_f1': 0.7671325727522289, 'weighted_precision': 0.8720554732179038, 'weighted_recall': 0.871236193070056, 'weighted_f1': 0.8714750012269527}
V1 routing: {'rows': 6609, 'auto_routed': 5092, 'human_review': 1517, 'coverage': 0.7704645180814041, 'review_rate': 0.22953548191859585, 'correct_routed': 4839, 'incorrect_routed': 253, 'routed_accuracy': 0.9503142183817753, 'misroute_rate': 0.049685781618224664}
V2 classification: {'accuracy': 0.8881827810561356, 'macro_precision': 0.8238285720809315, 'macro_recall': 0.7708454621896874, 'macro_f1': 0.7949299339848303, 'weighted_precision': 0.8852294243624294, 'weighted_recall': 0.8881827810561356, 'weighted_f1': 0.8858909203034717}
V2 routing: {'rows': 6609, 'auto_routed': 5404, 'human_review': 1205, 'coverage': 0.8176728703283401, 'review_rate': 0.18232712967165987, 'correct_routed': 5121, 'incorrect_routed': 283, 

## 4. Classification, routing, post-hoc diagnostics, and figures

Locked development-selected policies are the primary routing comparison. Final-test risk–coverage grids and fixed risk/coverage targets are descriptive post-hoc diagnostics only and cannot alter either policy.


In [4]:
def grid(y,p,top,margin):
    rows=[]
    for t in sorted(set(np.round(np.quantile(top,Q),2))):
        for m in sorted(set(np.round(np.quantile(margin,Q),2))):
            a=(top>=t)&(margin>=m)&(margin>0); n=int(a.sum())
            if n: rows.append({"Top threshold":float(t),"Margin threshold":float(m),"Coverage":float(n/len(y)),"Misroute rate":float(((y!=p)&a).sum()/n)})
    return pd.DataFrame(rows)
def matched(curve,name):
    risks=[]; covers=[]
    for lim in RISK_LIMITS:
        e=curve[curve["Misroute rate"]<=lim]
        cov=np.nan if e.empty else float(e.sort_values(["Coverage","Misroute rate","Top threshold","Margin threshold"],ascending=[False,True,True,True]).iloc[0]["Coverage"])
        risks.append({"Model":name,"Risk limit":lim,"Coverage":cov})
    for target in COVERAGE_TARGETS:
        c=curve.assign(distance=(curve["Coverage"]-target).abs()).sort_values(["distance","Misroute rate","Top threshold","Margin threshold"]).iloc[0]
        covers.append({"Model":name,"Coverage target":target,"Observed coverage":float(c["Coverage"]),"Misroute rate":float(c["Misroute rate"])})
    return pd.DataFrame(risks),pd.DataFrame(covers)

v1_curve=grid(y,v1_p,v1_top,v1_margin); v2_curve=grid(y,v2_p,v2_top,v2_margin)
a,b=matched(v1_curve,"Version 1"); c,d=matched(v2_curve,"Version 2")
matched_risk=pd.concat([a,c],ignore_index=True); matched_coverage=pd.concat([b,d],ignore_index=True)
class_cmp=pd.DataFrame([{"Model":"Version 1",**v1_m},{"Model":"Version 2",**v2_m}])
route_cmp=pd.DataFrame([{"Model":"Version 1",**v1_r},{"Model":"Version 2",**v2_r}])
cat_class=v1_cc.merge(v2_cc,on=["Category","Support"],suffixes=(" V1"," V2")); cat_class["F1 difference (V2 - V1)"]=cat_class["F1 V2"]-cat_class["F1 V1"]
cat_route=v1_cr.merge(v2_cr,on=["Category","Support"],suffixes=(" V1"," V2")); cat_route["Coverage difference (V2 - V1)"]=cat_route["Coverage V2"]-cat_route["Coverage V1"]; cat_route["Misroute difference (V2 - V1)"]=cat_route["Misroute rate V2"]-cat_route["Misroute rate V1"]

FIG_DIR.mkdir(parents=True,exist_ok=True); sns.set_theme(style="whitegrid")
fig,axes=plt.subplots(1,2,figsize=(19,8),sharex=True,sharey=True)
for ax,mat,title in zip(axes,(v1_cmn,v2_cmn),("Version 1: TF-IDF + Linear SVM","Version 2: DistilBERT")):
    sns.heatmap(mat,annot=True,fmt=".2f",cmap="Blues",vmin=0,vmax=1,xticklabels=SHORT,yticklabels=SHORT,square=True,cbar=False,ax=ax)
    ax.set_title(title); ax.set_xlabel("Predicted category"); ax.set_ylabel("Actual category"); ax.tick_params(axis="x",rotation=45); ax.tick_params(axis="y",rotation=0)
fig.suptitle("Shared 2024 Final Internal Benchmark: Row-Normalized Confusion Matrices",fontsize=15); fig.tight_layout(); fig.savefig(CM_FIG,dpi=200,bbox_inches="tight"); plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(16,6))
pd.DataFrame({"Metric":["Coverage","Review rate","Routed accuracy","Misroute rate"],"Version 1":[v1_r["coverage"],v1_r["review_rate"],v1_r["routed_accuracy"],v1_r["misroute_rate"]],"Version 2":[v2_r["coverage"],v2_r["review_rate"],v2_r["routed_accuracy"],v2_r["misroute_rate"]]}).set_index("Metric").plot(kind="bar",ax=axes[0],color=["#4C78A8","#F58518"])
axes[0].set_title("Locked Development-Selected Policies"); axes[0].set_ylabel("Rate"); axes[0].set_ylim(0,1); axes[0].tick_params(axis="x",rotation=25)
axes[1].scatter(v1_curve["Coverage"],v1_curve["Misroute rate"],alpha=.2,s=14,label="V1 diagnostic grid"); axes[1].scatter(v2_curve["Coverage"],v2_curve["Misroute rate"],alpha=.2,s=14,label="V2 diagnostic grid")
axes[1].scatter([v1_r["coverage"]],[v1_r["misroute_rate"]],marker="*",s=180,label="V1 locked policy"); axes[1].scatter([v2_r["coverage"]],[v2_r["misroute_rate"]],marker="*",s=180,label="V2 locked policy")
axes[1].set_title("Post-Hoc Final-Test Risk–Coverage Diagnostics"); axes[1].set_xlabel("Coverage"); axes[1].set_ylabel("Misroute rate"); axes[1].set_xlim(0,1); axes[1].set_ylim(bottom=0); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(ROUTING_FIG,dpi=200,bbox_inches="tight"); plt.close(fig)
print(class_cmp.to_string(index=False,float_format=lambda x:f"{x:.6f}")); print(cat_class.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
print(route_cmp.to_string(index=False,float_format=lambda x:f"{x:.6f}")); print(cat_route.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
print("Matched-risk diagnostics:"); print(matched_risk.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
print("Matched-coverage diagnostics:"); print(matched_coverage.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
print("Classification and routing figures: PASS")


    Model  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
Version 1  0.871236         0.773416      0.762083  0.767133            0.872055         0.871236     0.871475
Version 2  0.888183         0.823829      0.770845  0.794930            0.885229         0.888183     0.885891
                                           Category  Precision V1  Recall V1    F1 V1  Support  Precision V2  Recall V2    F1 V2  F1 difference (V2 - V1)
                        Checking or savings account      0.758929   0.794393 0.776256      428      0.823529   0.850467 0.836782                 0.060526
                                        Credit card      0.729323   0.765286 0.746872      507      0.777778   0.759369 0.768463                 0.021591
Credit reporting or other personal consumer reports      0.939070   0.932994 0.936022     4328      0.928699   0.960028 0.944104                 0.008081
                                    Debt collection

## 5. Hardware-specific compute comparison

Steady-state inference excludes loading time. DistilBERT timing includes tokenization, padding, device transfer, and synchronized inference. Results are local hardware measurements, not production service-level benchmarks.


In [5]:
sample=texts[:TIMING_N]; one=sample[0]
threads={"logical_cpu_count":os.cpu_count(),"torch_num_threads":torch.get_num_threads(),"torch_num_interop_threads":torch.get_num_interop_threads()}
def measure(fn,warm,reps,sync=False):
    for _ in range(warm): fn(); torch.cuda.synchronize() if sync else None
    out=[]
    for _ in range(reps):
        if sync: torch.cuda.synchronize()
        s=time.perf_counter(); fn()
        if sync: torch.cuda.synchronize()
        out.append(time.perf_counter()-s)
    return out
def v1_cpu_batch(items,batch_size):
    outputs=[]
    for start in range(0,len(items),batch_size):
        outputs.append(v1.predict(items[start:start+batch_size]))
    return outputs
v1_single=measure(lambda:v1.predict([one]),SINGLE_WARM,SINGLE_REPS)
v1_batch=measure(lambda:v1_cpu_batch(sample,BATCH),BATCH_WARM,BATCH_REPS)
v2.to("cpu").eval(); torch.cuda.empty_cache()
def v2_cpu(items,bs):
    with torch.inference_mode():
        return [v2(**tok(items[i:i+bs],truncation=True,max_length=256,padding=True,return_tensors="pt",return_token_type_ids=False)).logits for i in range(0,len(items),bs)]
v2_cpu_single=measure(lambda:v2_cpu([one],1),SINGLE_WARM,SINGLE_REPS)
v2_cpu_batch=measure(lambda:v2_cpu(sample,BATCH),BATCH_WARM,BATCH_REPS)
v2.to("cuda").eval(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def v2_gpu(items,bs):
    out=[]
    with torch.inference_mode():
        for i in range(0,len(items),bs):
            enc=tok(items[i:i+bs],truncation=True,max_length=256,padding=True,return_tensors="pt",return_token_type_ids=False); enc={k:x.cuda() for k,x in enc.items()}
            with torch.autocast(device_type="cuda",dtype=torch.float16): out.append(v2(**enc).logits)
    return out
v2_gpu_single=measure(lambda:v2_gpu([one],1),SINGLE_WARM,SINGLE_REPS,True)
v2_gpu_batch=measure(lambda:v2_gpu(sample,BATCH),BATCH_WARM,BATCH_REPS,True)
timing_alloc=torch.cuda.max_memory_allocated()/(1024**2); timing_reserved=torch.cuda.max_memory_reserved()/(1024**2)
med=lambda x:float(np.median(np.asarray(x)))
compute={"timing_sample_size":TIMING_N,"batch_size":BATCH,"single_warmups":SINGLE_WARM,"single_repetitions":SINGLE_REPS,"batch_warmups":BATCH_WARM,"batch_repetitions":BATCH_REPS,"cpu_threads":threads,"v1_load_seconds":v1_load,"v2_load_seconds":v2_load,"v1_rss_increase_mib":max(0,rss1-rss0)/(1024**2),"v2_rss_increase_mib":max(0,rss3-rss2)/(1024**2),"v1_artifact_bytes":V1_PATH.stat().st_size,"v2_artifact_bytes":sum((V2_DIR/n).stat().st_size for n in V2_FILES),"v1_cpu_single_median_ms":med(v1_single)*1000,"v2_cpu_single_median_ms":med(v2_cpu_single)*1000,"v1_cpu_batch_median_seconds":med(v1_batch),"v2_cpu_batch_median_seconds":med(v2_cpu_batch),"v1_cpu_batch_throughput":TIMING_N/med(v1_batch),"v2_cpu_batch_throughput":TIMING_N/med(v2_cpu_batch),"v2_gpu_single_median_ms":med(v2_gpu_single)*1000,"v2_gpu_batch_median_seconds":med(v2_gpu_batch),"v2_gpu_batch_throughput":TIMING_N/med(v2_gpu_batch),"v2_eval_peak_allocated_mib":eval_alloc,"v2_eval_peak_reserved_mib":eval_reserved,"v2_timing_peak_allocated_mib":timing_alloc,"v2_timing_peak_reserved_mib":timing_reserved,"hardware":{"os":platform.platform(),"cpu":platform.processor(),"ram_gib":round(psutil.virtual_memory().total/(1024**3),2),"gpu":torch.cuda.get_device_name(0),"gpu_memory_mib":round(torch.cuda.get_device_properties(0).total_memory/(1024**2)),"cuda_runtime":torch.version.cuda}}
def label_bars(ax,bars,formatter):
    for bar in bars:
        value=bar.get_height()
        ax.annotate(formatter(value),(bar.get_x()+bar.get_width()/2,value),xytext=(0,5),textcoords="offset points",ha="center",va="bottom",fontsize=9)
fig,axes=plt.subplots(2,2,figsize=(14,10))
artifact_values=[compute["v1_artifact_bytes"]/(1024**2),compute["v2_artifact_bytes"]/(1024**2)]
bars=axes[0,0].bar(["Version 1","Version 2"],artifact_values); axes[0,0].set_title("Serialized Artifact Size"); axes[0,0].set_ylabel("MiB"); axes[0,0].set_yscale("log"); label_bars(axes[0,0],bars,lambda x:f"{x:,.2f} MiB")
latency_values=[compute["v1_cpu_single_median_ms"],compute["v2_cpu_single_median_ms"],compute["v2_gpu_single_median_ms"]]
bars=axes[0,1].bar(["V1 CPU","V2 CPU","V2 GPU"],latency_values); axes[0,1].set_title("Single-Record Median Latency"); axes[0,1].set_ylabel("Milliseconds"); label_bars(axes[0,1],bars,lambda x:f"{x:,.2f} ms")
throughput_values=[compute["v1_cpu_batch_throughput"],compute["v2_cpu_batch_throughput"],compute["v2_gpu_batch_throughput"]]
bars=axes[1,0].bar(["V1 CPU","V2 CPU","V2 GPU"],throughput_values); axes[1,0].set_title("Batch Throughput (128 Shared Texts)"); axes[1,0].set_ylabel("Rows/second"); axes[1,0].set_yscale("log"); label_bars(axes[1,0],bars,lambda x:f"{x:,.2f}")
load_values=[compute["v1_load_seconds"],compute["v2_load_seconds"]]
bars=axes[1,1].bar(["Version 1","Version 2"],load_values); axes[1,1].set_title("Model Load Time (Excluded)"); axes[1,1].set_ylabel("Seconds"); label_bars(axes[1,1],bars,lambda x:f"{x:.3f} s")
fig.suptitle("Hardware-Specific Compute Comparison"); fig.text(.5,.01,"Local hardware-specific measurements; not production service-level benchmarks.",ha="center",fontsize=9); fig.tight_layout(rect=(0,.035,1,.96)); fig.savefig(COMPUTE_FIG,dpi=200,bbox_inches="tight"); plt.close(fig)
print("Compute results:")
for key,value in compute.items():
    print(f"- {key}: {value}")


Compute results:
- timing_sample_size: 128
- batch_size: 16
- single_warmups: 2
- single_repetitions: 10
- batch_warmups: 1
- batch_repetitions: 3
- cpu_threads: {'logical_cpu_count': 6, 'torch_num_threads': 6, 'torch_num_interop_threads': 6}
- v1_load_seconds: 0.4118464000057429
- v2_load_seconds: 0.058064800017746165
- v1_rss_increase_mib: 14.8203125
- v2_rss_increase_mib: 4.36328125
- v1_artifact_bytes: 3392109
- v2_artifact_bytes: 268806032
- v1_cpu_single_median_ms: 1.3079999916953966
- v2_cpu_single_median_ms: 36.519350003800355
- v1_cpu_batch_median_seconds: 0.037183100008405745
- v2_cpu_batch_median_seconds: 8.545381799980532
- v1_cpu_batch_throughput: 3442.424111251183
- v2_cpu_batch_throughput: 14.9788509157416
- v2_gpu_single_median_ms: 38.79140000208281
- v2_gpu_batch_median_seconds: 8.2731868000119
- v2_gpu_batch_throughput: 15.471668063848854
- v2_eval_peak_allocated_mib: 441.9052734375
- v2_eval_peak_reserved_mib: 470.0
- v2_timing_peak_allocated_mib: 442.66064453125
- v

## 6. Aggregate local summary and final validation

The local JSON contains aggregate results only and supports the separately committed comparison report.


In [6]:
def records(frame):
    result=[]
    for row in frame.to_dict("records"):
        clean={}
        for k,v in row.items():
            if isinstance(v,np.integer): clean[k]=int(v)
            elif isinstance(v,np.floating): clean[k]=None if np.isnan(v) else float(v)
            else: clean[k]=v
        result.append(clean)
    return result
summary={"rows":6609,"policy_path":POLICY_PATH.relative_to(ROOT).as_posix(),"policy_sha256":policy_sha,"policy":policy,"v1_classification":v1_m,"v2_classification":v2_m,"v1_routing":v1_r,"v2_routing":v2_r,"v1_category_classification":records(v1_cc),"v2_category_classification":records(v2_cc),"category_classification_comparison":records(cat_class),"v1_category_routing":records(v1_cr),"v2_category_routing":records(v2_cr),"category_routing_comparison":records(cat_route),"matched_risk_diagnostics":records(matched_risk),"matched_coverage_diagnostics":records(matched_coverage),"compute":compute,"row_output_path":ROW_PATH.relative_to(ROOT).as_posix(),"row_output_size_bytes":ROW_PATH.stat().st_size,"row_output_sha256":row_sha,"figures":[x.relative_to(ROOT).as_posix() for x in (CM_FIG,ROUTING_FIG,COMPUTE_FIG)],"restrictions":{"models_retrained":False,"2025_or_2026_accessed":False,"final_test_used_for_policy_selection":False}}
SUMMARY_PATH.write_text(json.dumps(summary,indent=2,sort_keys=True),encoding="utf-8")
staged=set(subprocess.check_output(["git","diff","--cached","--name-only"],cwd=ROOT,text=True).splitlines())
source=NOTEBOOK_PATH.read_text(encoding="utf-8")
notebook_object=json.loads(source)
audited_source="".join("".join(cell.get("source",[])) for cell in notebook_object["cells"][:-1])
valid={"OOF-only policy":policy["oof_rows"]==26433 and policy["final_test_data_loaded_at_lock"] is False,"Policy fingerprint":sha(POLICY_PATH)==policy_sha,"No training calls":".fit(" not in audited_source and ".fit_transform(" not in audited_source and ".train(" not in audited_source,"Locked source only":ACCESSED=={DATA.resolve()},"Rows and overlap":len(dev_idx)==26433 and len(y)==6609 and overlap==0,"Same benchmark labels":v1_p.shape==v2_p.shape==y.shape,"V1 metrics reproduced":all(round(v1_m[n],4)==V1_REF[n] for n in ("accuracy","macro_precision","macro_recall","macro_f1","weighted_precision","weighted_recall","weighted_f1")),"V1 routing reproduced":all(round(v1_r[n],4)==V1_REF[n] for n in ("coverage","routed_accuracy","misroute_rate")),"V2 finite":np.isfinite(v2_logits).all(),"Local outputs ignored":all(ignored(x) for x in (POLICY_PATH,ROW_PATH,SUMMARY_PATH)),"No model output staged":not any(x.startswith("models/") for x in staged),"Figures created":all(x.is_file() and x.stat().st_size>0 for x in (CM_FIG,ROUTING_FIG,COMPUTE_FIG)),"No 2025/2026 paths":"cfpb_complaints_2025" not in audited_source and "cfpb_complaints_2026" not in audited_source}
failed=[k for k,v in valid.items() if not v]
print(pd.DataFrame([(k,"PASS" if v else "FAIL") for k,v in valid.items()],columns=["Check","Result"]).to_string(index=False))
print(f"Final Issue #41 notebook validation: {len(valid)-len(failed)}/{len(valid)} checks passed")
if failed: raise RuntimeError(failed)
print(f"Aggregate summary SHA-256: {sha(SUMMARY_PATH)}")
print("Neither model was retrained; no 2025/2026 data was accessed; no row-level outputs were displayed.")
print("Issue #41 notebook execution: PASS")


                 Check Result
       OOF-only policy   PASS
    Policy fingerprint   PASS
     No training calls   PASS
    Locked source only   PASS
      Rows and overlap   PASS
 Same benchmark labels   PASS
 V1 metrics reproduced   PASS
 V1 routing reproduced   PASS
             V2 finite   PASS
 Local outputs ignored   PASS
No model output staged   PASS
       Figures created   PASS
    No 2025/2026 paths   PASS
Final Issue #41 notebook validation: 13/13 checks passed
Aggregate summary SHA-256: 8f185a5a633683ed46aca73e989c689faa71023281735e398266ed1e4670d3ce
Neither model was retrained; no 2025/2026 data was accessed; no row-level outputs were displayed.
Issue #41 notebook execution: PASS
